# 10 - RecBole Hyperparameter Optimization (Sequential Models)

This notebook performs hyperparameter optimization for RecBole **sequential** recommendation models.

**Models:**
- SASRec, GRU4Rec, NARM, SRGNN
- BERT4Rec excluded: masked item prediction incompatible with pre-augmented data (NaN loss)

**Prerequisite:** Run notebook 09 first to create the pre-augmented sequential data.

**Data:** Uses `redial_seq` dataset with pre-augmented `item_id_list` columns
(bypasses RecBole bug [#1593](https://github.com/RUCAIBox/RecBole/issues/1593)).

**Outputs:**
- `data/recbole/hpo_results_seq/best_hyperparameters.json`

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

# PyTorch 2.x compatibility fix for RecBole checkpoints
import torch

if not getattr(torch, "_recbole_patched", False):
    _original_load = torch.load

    def _patched_load(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return _original_load(*args, **kwargs)

    torch.load = _patched_load
    torch._recbole_patched = True
    print("Applied weights_only=False patch for RecBole compatibility")
else:
    print("PyTorch patch already applied (skipping)")

print(f"PyTorch version: {torch.__version__}")

In [ ]:
import json
import warnings
from itertools import product
from pathlib import Path

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import get_model, get_trainer, init_seed
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# Configuration
DATA_PATH = Path("../data")
RECBOLE_DATA_PATH = DATA_PATH / "recbole"
RESULTS_PATH = DATA_PATH / "recbole" / "hpo_results_seq"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

SEED = 42

In [ ]:
def get_device() -> str:
    """Get best available device: CUDA > CPU (RecBole does not support MPS)."""
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


DEVICE = get_device()
print(f"Using device: {DEVICE}")

## Base Configuration

Key differences from CF config (notebook 07):
- `dataset`: `redial_seq` (pre-augmented sequential format)
- `load_col`: includes `item_id_list` instead of `timestamp`
- `alias_of_item_id`: `["item_id_list"]` — shared item embedding space
- `repeatable`: `True` — required for sequential models
- No negative sampling (all sequential models use CE loss)

In [ ]:
# Ensure saved/ directory exists for RecBole checkpoints
import shutil
from pathlib import Path

for cache_dir in ["dataset", "log"]:
    if Path(cache_dir).exists():
        shutil.rmtree(cache_dir)
        print(f"Cleared {cache_dir}/")
Path("saved").mkdir(exist_ok=True)
print("saved/ directory ready")

In [ ]:
BASE_CONFIG = {
    # Dataset settings (sequential)
    "data_path": str(RECBOLE_DATA_PATH),
    "dataset": "redial_seq",
    "benchmark_filename": ["train", "valid", "test"],
    # Field definitions
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "LIST_SUFFIX": "_list",
    "ITEM_LIST_LENGTH_FIELD": "item_length",
    "load_col": {
        "inter": ["user_id", "item_id", "item_id_list"],
    },
    # Sequential-specific settings
    "MAX_ITEM_LIST_LENGTH": 50,
    "alias_of_item_id": ["item_id_list"],
    "repeatable": True,
    # Training settings
    "epochs": 100,
    "train_batch_size": 2048,
    "eval_batch_size": 2048,
    "learning_rate": 0.001,
    "stopping_step": 5,
    # Evaluation settings
    "eval_args": {
        "group_by": "user",
        "order": "TO",
        "split": {"LS": "valid_and_test"},
        "mode": "full",
    },
    "metrics": ["Recall", "MRR", "NDCG", "Hit", "Precision"],
    "topk": [1, 5, 10],
    "valid_metric": "NDCG@10",
    # No negative sampling (CE loss models)
    "train_neg_sample_args": None,
    # Device and reproducibility
    "device": DEVICE,
    "seed": SEED,
    "reproducibility": True,
    "show_progress": True,
    # Logging
    "log_wandb": False,
    "state": "INFO",
}

## Model-Specific Hyperparameter Grids

In [ ]:
HYPERPARAMETER_GRIDS = {
    "SASRec": {
        "n_layers": [1, 2, 3],
        "n_heads": [1, 2],
        "hidden_dropout_prob": [0.2, 0.5],
        "attn_dropout_prob": [0.2, 0.5],
        "learning_rate": [0.01, 0.001, 0.0001],
    },
    "GRU4Rec": {
        "hidden_size": [128],
        "num_layers": [1, 2, 3],
        "dropout_prob": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
        "learning_rate": [0.01, 0.001, 0.0001],
    },
    # BERT4Rec dropped: uses masked item prediction which is incompatible
    # with our pre-augmented sliding-window data format (causes NaN loss)
    "NARM": {
        "hidden_size": [64, 128],
        "n_layers": [1, 2],
        "dropout_probs": [[0.1, 0.1], [0.3, 0.3]],
        "learning_rate": [0.001, 0.0005, 0.0001],
    },
    "SRGNN": {
        "step": [1, 2],
        "learning_rate": [0.01, 0.001, 0.0001],
    },
}

SEQUENTIAL_MODELS = ["SASRec", "GRU4Rec", "NARM", "SRGNN"]

# Print grid sizes
print("Hyperparameter grid sizes:")
for model, grid in HYPERPARAMETER_GRIDS.items():
    n_combos = 1
    for values in grid.values():
        n_combos *= len(values)
    print(f"  {model}: {n_combos} combinations")

print(f"\nSequential models: {SEQUENTIAL_MODELS}")

## HPO Functions

In [ ]:
def run_single_experiment(
    model_name: str,
    hyperparams: dict,
    base_config: dict,
) -> dict:
    """Run a single training experiment and return results."""
    config_dict = base_config.copy()
    config_dict.update(hyperparams)
    config_dict["model"] = model_name

    try:
        config = Config(model=model_name, config_dict=config_dict)
        init_seed(config["seed"], config["reproducibility"])

        dataset = create_dataset(config)
        train_data, valid_data, test_data = data_preparation(config, dataset)

        model = get_model(config["model"])(config, train_data._dataset).to(
            config["device"]
        )
        trainer = get_trainer(config["MODEL_TYPE"], config["model"])(config, model)

        best_valid_score, best_valid_result = trainer.fit(
            train_data, valid_data, verbose=False, show_progress=False
        )

        test_result = trainer.evaluate(test_data)

        return {
            "status": "success",
            "best_valid_score": float(best_valid_score),
            "valid_result": {k: float(v) for k, v in best_valid_result.items()},
            "test_result": {k: float(v) for k, v in test_result.items()},
        }
    except Exception as e:
        print(f"    ERROR: {e}")
        import traceback

        traceback.print_exc()
        return {
            "status": "failed",
            "error": str(e),
            "best_valid_score": -float("inf"),
        }

In [ ]:
def grid_search(
    model_name: str,
    param_grid: dict,
    base_config: dict,
) -> dict:
    """Perform grid search over hyperparameter combinations."""
    if not param_grid:
        print(f"  No hyperparameters to tune for {model_name}")
        result = run_single_experiment(model_name, {}, base_config)
        return {
            "best_params": {},
            "best_valid_score": result.get("best_valid_score", -float("inf")),
            "best_result": result,
            "all_results": [result],
        }

    param_names = list(param_grid.keys())
    param_values = list(param_grid.values())
    combinations = list(product(*param_values))

    print(f"  Testing {len(combinations)} parameter combinations")

    best_score = -float("inf")
    best_params = None
    best_result = None
    all_results = []

    for combo in tqdm(combinations, desc=f"  {model_name}", leave=False):
        params = dict(zip(param_names, combo))
        result = run_single_experiment(model_name, params, base_config)
        result["params"] = params
        all_results.append(result)

        if result["status"] == "success":
            score = result["best_valid_score"]
            if score > best_score:
                best_score = score
                best_params = params
                best_result = result

    return {
        "best_params": best_params,
        "best_valid_score": best_score,
        "best_result": best_result,
        "all_results": all_results,
    }

## Run HPO for Sequential Models

In [ ]:
hpo_results = {}

print("=" * 60)
print("SEQUENTIAL MODELS")
print("=" * 60)

for model_name in SEQUENTIAL_MODELS:
    print(f"\n{model_name}:")
    param_grid = HYPERPARAMETER_GRIDS.get(model_name, {})

    result = grid_search(model_name, param_grid, BASE_CONFIG)
    hpo_results[model_name] = result

    if result["best_params"]:
        print(f"  Best params: {result['best_params']}")
    print(f"  Best valid NDCG@10: {result['best_valid_score']:.4f}")

## Save Results

In [ ]:
best_hyperparams = {}
for model_name, result in hpo_results.items():
    best_hyperparams[model_name] = {
        "params": result["best_params"] or {},
        "valid_ndcg@10": result["best_valid_score"],
        "test_result": result.get("best_result", {}).get("test_result", {}),
    }

with open(RESULTS_PATH / "best_hyperparameters.json", "w") as f:
    json.dump(best_hyperparams, f, indent=2)

print(f"Saved best hyperparameters to {RESULTS_PATH / 'best_hyperparameters.json'}")

In [ ]:
print("\n" + "=" * 60)
print("HPO SUMMARY (Sequential Models)")
print("=" * 60)

print(f"\n{'Model':<15} {'Valid NDCG@10':>15} {'Best Params'}")
print("-" * 60)

sorted_models = sorted(
    best_hyperparams.items(),
    key=lambda x: x[1]["valid_ndcg@10"],
    reverse=True,
)

for model_name, hp in sorted_models:
    score = hp["valid_ndcg@10"]
    params_str = str(hp["params"]) if hp["params"] else "(default)"
    if len(params_str) > 40:
        params_str = params_str[:37] + "..."
    print(f"{model_name:<15} {score:>15.4f} {params_str}")

In [ ]:
print("\nHPO complete! Ready for sequential evaluation (notebook 11).")